# Entrenamiento con el train completo (~1.93M imágenes) — Colab

Notebook delgado: todo el entrenamiento vive en `src/entrenamiento` (el mismo código de la prueba local de 200k).
Acá solo se prepara la máquina y se lanza `scripts/entrenar.py --config configs/completo_2m.json`.

**Para empezar o retomar: Ejecutar todo.** La corrida guarda todo en Drive (`carpeta_corrida`) y sigue sola desde
el último checkpoint. Si la sesión se corta, se reconecta y se vuelve a ejecutar todo: se vuelven a bajar los
parquets (el disco de la máquina se borra) y el entrenamiento retoma donde quedó.

**Para pausar a mano:** interrumpir la celda de entrenamiento (■). El script recibe la señal, guarda un checkpoint
y sale. `MAX_HORAS` hace lo mismo solo, antes de que Colab corte la sesión.

Por qué se bajan todos los parquets al disco local en vez de leerlos de HF de a uno: cada ronda es balanceada por
celda, así que toca casi todos los parquets; leerlos de HF en cada ronda sería bajar ~180 GB por ronda.

In [ ]:
# ---- Parámetros de la sesión ----
RAMA = "pipeline-2m-entrenamiento"
DRIVE = "/content/drive/MyDrive/geo-project"     # Carpeta de Drive del proyecto.
INSUMOS_DRIVE = f"{DRIVE}/insumos"                # manifest.parquet, splits.parquet, validation.parquet (subidos a mano).
CARPETA_CORRIDA = f"{DRIVE}/entrenamiento_2m/completo_2m"
MAX_HORAS = 20.0    # Por sesión. Por debajo del límite de Colab (24 h en Pro) para que guarde y salga solo.
CONFIG = "configs/completo_2m.json"
REPO = "/content/geo-project"
SHARDS = "/content/shards"                        # Disco local de la máquina (rápido, se borra al cerrar la sesión).

## 1. Máquina

In [ ]:
import os, shutil, subprocess
!nvidia-smi --query-gpu=name,memory.total --format=csv
print("CPUs:", os.cpu_count())
!free -g | head -2
total, usado, libre = shutil.disk_usage("/content")
print(f"Disco /content: {libre / 1e9:.0f} GB libres de {total / 1e9:.0f} GB")
assert libre > 200e9, "Hacen falta ~185 GB para los parquets de train y val (+ margen)." 

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(CARPETA_CORRIDA, exist_ok=True)
for nombre in ["manifest.parquet", "splits.parquet", "validation.parquet"]:
    assert os.path.exists(f"{INSUMOS_DRIVE}/{nombre}"), f"Falta subir {nombre} a {INSUMOS_DRIVE}" 

## 2. Código

In [ ]:
if os.path.exists(REPO):
    !git -C {REPO} fetch -q origin {RAMA} && git -C {REPO} checkout -q {RAMA} && git -C {REPO} pull -q --ff-only
else:
    !git clone -q --branch {RAMA} https://github.com/gggelme/geo-project.git {REPO}
!git -C {REPO} log -1 --oneline

## 3. Datos al disco local

In [ ]:
# Insumos chicos: de Drive al disco local (leerlos de Drive en cada ronda sería lento).
INSUMOS = "/content/insumos"
os.makedirs(INSUMOS, exist_ok=True)
for nombre in ["manifest.parquet", "splits.parquet", "validation.parquet"]:
    if not os.path.exists(f"{INSUMOS}/{nombre}"):
        shutil.copy(f"{INSUMOS_DRIVE}/{nombre}", f"{INSUMOS}/{nombre}")
print(os.listdir(INSUMOS))

In [ ]:
# Parquets de train y val (no los de test), en la revisión fijada en la config. Se reanuda si se corta.
import json, time
import pandas as pd
from huggingface_hub import snapshot_download

cfg_json = json.load(open(f"{REPO}/{CONFIG}"))
splits = pd.read_parquet(f"{INSUMOS}/splits.parquet", columns=["split", "archivo"])
archivos = sorted(splits.loc[splits.split.isin(["train", "val"]), "archivo"].unique())
print(f"{len(archivos)} parquets a bajar")
t0 = time.time()
snapshot_download(cfg_json.get("hf_repo", "josefbednar/world-streetview-500k"), repo_type="dataset",
                  revision=cfg_json["hf_revision"], local_dir=SHARDS,
                  allow_patterns=[f"data/{a}" for a in archivos], max_workers=8)
faltan = [a for a in archivos if not os.path.exists(f"{SHARDS}/data/{a}")]
assert not faltan, f"Faltan {len(faltan)} parquets: {faltan[:5]}"
gb = sum(os.path.getsize(f"{SHARDS}/data/{a}") for a in archivos) / 1e9
print(f"Listo: {gb:.0f} GB en {(time.time() - t0) / 60:.1f} min")

## 4. Entrenamiento

Retoma solo si `CARPETA_CORRIDA` ya tiene checkpoints. La salida se ve en vivo; además todo queda en
`log.jsonl` e `historial.csv` dentro de la carpeta de la corrida.

In [ ]:
import signal, sys
num_workers = max(2, (os.cpu_count() or 2) - 1)
comando = [sys.executable, "scripts/entrenar.py", "--config", CONFIG, "--cambiar",
           f"carpeta_corrida={CARPETA_CORRIDA}", f"datos_local={SHARDS}/data",
           f"manifest={INSUMOS}/manifest.parquet", f"splits={INSUMOS}/splits.parquet",
           f"val_global={INSUMOS}/validation.parquet", f"num_workers={num_workers}", f"max_horas={MAX_HORAS}"]
print(" ".join(comando))
proceso = subprocess.Popen(comando, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
try:
    for linea in proceso.stdout:
        print(linea, end="")
except KeyboardInterrupt:
    # SIGTERM: el script termina el paso actual, guarda un checkpoint y sale (puede tardar si está evaluando).
    print("Interrumpido: se pide al script que guarde y salga (esperar, no interrumpir de nuevo)...")
    proceso.send_signal(signal.SIGTERM)
    while proceso.poll() is None:
        try:
            for linea in proceso.stdout:
                print(linea, end="")
        except KeyboardInterrupt:
            print("Sigue guardando; esperando a que salga...")
codigo = proceso.wait()
print({0: "TERMINADO", 3: "PAUSADO: volver a ejecutar todo para seguir"}.get(codigo, f"ERROR (código {codigo})"))

## 5. Cómo va

In [ ]:
ruta = f"{CARPETA_CORRIDA}/historial.csv"
if os.path.exists(ruta):
    h = pd.read_csv(ruta)
    display(h[["ronda", "loss_train", "val_ce_haversine", "val_top1", "val_top5", "val_km_mediana",
               "valglobal_top1", "valglobal_km_mediana", "lr_cabeza", "horas_acumuladas"]].tail(15))